# MIMIC-Sepsis Benchmark — Kaggle Edition

**Stage 4 only.** This notebook assumes you have already generated a `patient_timeseries_*.csv`
from Stages 1–3 (locally or via a separate Kaggle notebook) and uploaded it as a Kaggle dataset.

### Required Kaggle datasets (attach before running)
| Kaggle dataset slug | Contents |
|---|---|
| any name | All `.py` files from the project's `src/` directory |
| any name | Your `patient_timeseries_*.csv` from `processed_files/` |

The notebook **auto-detects** the dataset paths — no hardcoded slugs needed.
See `plans/kaggle_migration.md` in this repo for step-by-step upload instructions.

In [ ]:
# ── Cell 1: Install packages not pre-installed on Kaggle ─────────────────────
# torch, scikit-learn, xgboost, pandas, numpy, matplotlib, seaborn are
# already available in Kaggle's base image.
# The packages below are the additional ones this project requires.
# NOTE: pyprind is no longer needed — data_processor.py uses a built-in
#       progress bar that cannot be monkey-patched.

# Uninstall the Kaggle system 'benchmark' package which would otherwise
# shadow our src/benchmark.py when imported by name.
!pip uninstall -q -y benchmark 2>/dev/null; echo 'system benchmark removed (or was not installed)'
!pip install -q lightgbm prophet pytorch-forecasting

# Verify key imports
import importlib
for pkg in ['lightgbm', 'prophet', 'pytorch_forecasting']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError as e:
        print(f'  ✗ {pkg}: {e}')

In [ ]:
# ── Cell 2: Auto-detect src/ path and load modules by file ───────────────────
#
# Kaggle datasets are mounted under /kaggle/input/ at paths that depend on
# the owner username and dataset slug, which may vary.  Instead of hardcoding
# a path, we scan /kaggle/input/ for the directory that contains benchmark.py.
#
# We also use importlib.util.spec_from_file_location to load our benchmark
# module BY FILE PATH rather than by name.  This completely bypasses any
# system-level 'benchmark' package that Kaggle may have pre-installed.

import sys
import os
import importlib
import importlib.util

# ── 1. Find the directory that contains benchmark.py ─────────────────────────
def find_src_dir(root="/kaggle/input", target="benchmark.py"):
    """Walk /kaggle/input and return the first directory containing target."""
    for dirpath, dirnames, filenames in os.walk(root):
        if target in filenames:
            return dirpath
    return None

SRC_DIR = find_src_dir()
if SRC_DIR is None:
    raise FileNotFoundError(
        "Could not find benchmark.py under /kaggle/input/. "
        "Make sure you have attached the mimic-sepsis-src dataset containing "
        "all .py files from the src/ directory."
    )
print(f"Found src/ at: {SRC_DIR}")

# ── 2. Insert at the FRONT of sys.path so our files take priority ─────────────
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# ── Evict stale cached modules from any previous kernel run ──────────────────
# data_processor and benchmark are loaded by file path below, but if a
# previous run already registered them in sys.modules we flush them first
# so the load_module_from_file call always executes fresh code.
import importlib
for _mod in list(sys.modules.keys()):
    if _mod in ('data_processor', 'benchmark') or _mod.startswith('benchmark.'):
        del sys.modules[_mod]

# ── 3. Load benchmark.py by absolute file path (bypasses name collision) ──────
def load_module_from_file(module_name, file_path):
    """Load a Python file as a module regardless of sys.path shadowing."""
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod   # register before exec so relative imports resolve
    spec.loader.exec_module(mod)
    return mod

bm = load_module_from_file("benchmark", os.path.join(SRC_DIR, "benchmark.py"))

run_benchmark          = bm.run_benchmark
run_all_experiments    = bm.run_all_experiments
run_selected_experiments = bm.run_selected_experiments
set_random_seeds       = bm.set_random_seeds

# ── 4. Working directory ──────────────────────────────────────────────────────
WORK_DIR = "/kaggle/working"
os.makedirs(os.path.join(WORK_DIR, "results"), exist_ok=True)
os.chdir(WORK_DIR)

# ── 5. Standard library imports ───────────────────────────────────────────────
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

set_random_seeds(42)
print(f"cwd     : {os.getcwd()}")
print(f"GPU     : {torch.cuda.is_available()}")
print("All imports OK.")

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────

# ── Data path ─────────────────────────────────────────────────────────────────
# Set this to the exact path of your patient_timeseries CSV on Kaggle.
# Example: "/kaggle/input/mimic-sepsis-data/patient_timeseries_2026-06-11-12-00-00.csv"
DATA_PATH = "/kaggle/input/mimic-sepsis-data/patient_timeseries_v4.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"CSV not found: {DATA_PATH}\n"
        "Update DATA_PATH above to the correct path.  "
        "You can list available files with:\n"
        "  import os; print(os.listdir('/kaggle/input/mimic-sepsis-data/'))"
    )
print(f"Data CSV : {DATA_PATH}")

# ── Task (prediction target) ───────────────────────────────────────────────────
# Options: 'morta_hosp' | 'los' | 'septic_shock' | 'vasopressor'
#          'sofa_score' | 'sirs_score' | 'news2_score'
TASK = "septic_shock"

# ── Model ──────────────────────────────────────────────────────────────────────
# Options: 'linear' | 'lstm' | 'transformer' | 'xgboost' | 'lightgbm'
#          'lstm_multistep' | 'prophet' | 'tft'
MODEL_TYPE = "transformer"

# ── Temporal prediction horizon (timesteps ahead; ignored for static tasks) ────
HORIZON = 6

# ── Include treatment variables as features? ───────────────────────────────────
INCLUDE_TREATMENTS = False

# ── Window-level class balancing ───────────────────────────────────────────────
BALANCE          = False          # True to enable
BALANCE_STRATEGY = "undersample"  # 'undersample' | 'oversample' | 'combined'

# ── Score thresholds (used when TASK is a score task) ─────────────────────────
SOFA_THRESHOLD  = 2
SIRS_THRESHOLD  = 2
NEWS2_THRESHOLD = 5

# ── Output CSV for results ─────────────────────────────────────────────────────
OUTPUT_CSV = os.path.join(WORK_DIR, "results", "score_benchmark.csv")

TEMPORAL_TASKS = [
    'septic_shock', 'mechvent', 'sepsis', 'vasopressor',
    'sofa_score', 'sirs_score', 'news2_score',
]

print(f"Task     : {TASK}")
print(f"Model    : {MODEL_TYPE}")
print(f"Horizon  : {HORIZON if TASK in TEMPORAL_TASKS else 'N/A (static task)'}")

In [ ]:
# ── Cell 3b: Memory optimisation (run before Cell 4 if you get OOM errors) ───
#
# Kaggle notebooks have 16 GB RAM.  Large patient_timeseries CSVs can exceed
# this when pandas uses float64/int64 by default.  This cell:
#   1. Reads the CSV in chunks, downcasting numeric columns to float32/int32.
#   2. Streams each chunk directly to disk — avoiding a full in-memory copy.
#   3. Optionally sub-samples to MAX_PATIENTS patients (requires a second pass
#      if set, but the first pass is already on the optimised file).
#   4. Updates DATA_PATH for subsequent cells.
#
# If your CSV fits comfortably in RAM, skip this cell entirely.

import gc

# ── Tuning knobs ──────────────────────────────────────────────────────────────
MAX_PATIENTS  = None   # e.g. 5000 to sub-sample; None = use all patients
CHUNK_SIZE    = 200_000  # rows per chunk when reading

OPTIMISED_CSV = os.path.join(WORK_DIR, "patient_timeseries_opt.csv")

def _downcast_df(df: pd.DataFrame) -> pd.DataFrame:
    """Cast float64→float32 and int64→int32 to halve memory footprint."""
    for col in df.select_dtypes('float64').columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes('int64').columns:
        df[col] = df[col].astype('int32')
    return df

# Stream chunks straight to disk — never hold more than one chunk + the open
# file handle in memory at the same time.
print(f"Reading {DATA_PATH} in chunks of {CHUNK_SIZE:,} rows ...")
total_rows = 0
if os.path.exists(OPTIMISED_CSV):
    os.remove(OPTIMISED_CSV)
for i, chunk in enumerate(pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE)):
    chunk = _downcast_df(chunk)
    total_rows += len(chunk)
    chunk.to_csv(OPTIMISED_CSV, mode='a', index=False, header=(i == 0))
    del chunk; gc.collect()
    print(f"  chunk {i+1}: {total_rows:,} rows written", end='\r')
print(f"\nStreamed {total_rows:,} rows → {OPTIMISED_CSV}")

# Optional: sub-sample patients (second pass on the already-small optimised file)
if MAX_PATIENTS is not None:
    df_ids = pd.read_csv(OPTIMISED_CSV, usecols=['stay_id'])
    all_ids = df_ids['stay_id'].unique()
    del df_ids; gc.collect()
    rng = np.random.default_rng(42)
    sampled_ids = set(rng.choice(all_ids, size=min(MAX_PATIENTS, len(all_ids)), replace=False).tolist())
    SUBSAMPLED_CSV = os.path.join(WORK_DIR, "patient_timeseries_sub.csv")
    wrote_header = False
    for chunk in pd.read_csv(OPTIMISED_CSV, chunksize=CHUNK_SIZE):
        sub = chunk[chunk['stay_id'].isin(sampled_ids)]
        if len(sub):
            sub.to_csv(SUBSAMPLED_CSV, mode='a', index=False, header=not wrote_header)
            wrote_header = True
        del chunk, sub; gc.collect()
    OPTIMISED_CSV = SUBSAMPLED_CSV
    print(f"Sub-sampled to {MAX_PATIENTS:,} patients → {OPTIMISED_CSV}")

# Point subsequent cells at the optimised file
DATA_PATH = OPTIMISED_CSV
print(f"DATA_PATH updated → {DATA_PATH}")

In [ ]:
# ── Cell 4: Single benchmark run ──────────────────────────────────────────────
# Trains one model on one task and prints AUROC / AUPRC (classification)
# or RMSE / MAE (regression).  Results are also appended to OUTPUT_CSV.

_score_thresholds = {
    'sofa_score':  SOFA_THRESHOLD,
    'sirs_score':  SIRS_THRESHOLD,
    'news2_score': NEWS2_THRESHOLD,
}

result = run_benchmark(
    task               = TASK,
    model_type         = MODEL_TYPE,
    include_treatments = INCLUDE_TREATMENTS,
    prediction_horizon = HORIZON if TASK in TEMPORAL_TASKS else None,
    balance            = BALANCE,
    balance_strategy   = BALANCE_STRATEGY,
    data_path          = DATA_PATH,
    score_thresholds   = _score_thresholds,
)

result_df = pd.DataFrame([result])
display(result_df.T.rename(columns={0: 'value'}))

write_header = not os.path.exists(OUTPUT_CSV)
result_df.to_csv(OUTPUT_CSV, mode='a', index=False, header=write_header)
print(f"\nResult appended to {OUTPUT_CSV}")

In [ ]:
# ── Cell 5a: Run all models for one task (optional) ───────────────────────────
# Iterates over linear / lstm / transformer across all prediction horizons
# for the task set in TASK above.  Results saved to {TASK}_benchmark_results.csv

# Uncomment to run:
# run_selected_experiments(
#     task               = TASK,
#     include_treatments = INCLUDE_TREATMENTS,
#     balance            = BALANCE,
#     balance_strategy   = BALANCE_STRATEGY,
#     score_thresholds   = _score_thresholds,
# )
print("Cell 5a: uncomment the block above to run.")

In [ ]:
# ── Cell 5b: Run ALL experiments (all tasks × all models) (optional) ──────────
# Long-running cell (~hours on CPU, faster with GPU for LSTM/Transformer).
# Results saved incrementally to benchmark_results.csv in WORK_DIR.

# Uncomment to run:
# run_all_experiments()
print("Cell 5b: uncomment the block above to run the full experiment grid.")

In [ ]:
# ── Cell 6: AUROC / AUPRC bar charts across models and tasks ─────────────────
#
# Reads the accumulated results CSV and produces:
#   • Figure 1: AUROC and AUPRC bar charts (classification tasks)
#   • Figure 2: RMSE and MAE bar charts    (regression tasks)
#
# Change RESULTS_CSV to point at run_all / run_selected output if needed.

RESULTS_CSV = OUTPUT_CSV

if not os.path.exists(RESULTS_CSV):
    print(f"Results file not found: {RESULTS_CSV}")
    print("Run Cell 4 (single run) or Cell 5 (multi-run) first.")
else:
    results_df = pd.read_csv(RESULTS_CSV)
    print(f"Loaded {len(results_df)} rows from {RESULTS_CSV}")
    display(results_df)

    CLASSIF_TASKS = ['morta_hosp', 'septic_shock', 'vasopressor', 'mechvent', 'sepsis']
    REGRESS_TASKS = ['los', 'sofa_score', 'sirs_score', 'news2_score']

    classif_df = results_df[results_df['task'].isin(CLASSIF_TASKS)].copy()
    regress_df = results_df[results_df['task'].isin(REGRESS_TASKS)].copy()

    MODEL_ORDER  = ['linear', 'lstm', 'transformer', 'xgboost', 'lightgbm']
    PALETTE      = sns.color_palette("Set2", n_colors=len(MODEL_ORDER))
    MODEL_COLORS = {m: PALETTE[i] for i, m in enumerate(MODEL_ORDER)}

    def _bar_chart(ax, df, metric_col, title, ylabel, higher_better=True):
        """Grouped bar chart: tasks on x-axis, models as groups."""
        if df.empty or metric_col not in df.columns:
            ax.set_visible(False)
            return
        tasks   = sorted(df['task'].unique())
        models  = [m for m in MODEL_ORDER if m in df['model_type'].unique()]
        width   = 0.8 / max(len(models), 1)
        x       = np.arange(len(tasks))
        agg_fn  = 'max' if higher_better else 'min'

        for i, model in enumerate(models):
            sub  = df[df['model_type'] == model]
            vals = []
            for t in tasks:
                row = sub[sub['task'] == t]
                if row.empty or metric_col not in row.columns:
                    vals.append(np.nan)
                else:
                    vals.append(getattr(row[metric_col], agg_fn)())
            offset = (i - len(models) / 2 + 0.5) * width
            bars = ax.bar(
                x + offset, vals, width,
                label=model,
                color=MODEL_COLORS.get(model, 'grey'),
                edgecolor='white', linewidth=0.5,
            )
            for bar, val in zip(bars, vals):
                if not np.isnan(val):
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.005,
                        f"{val:.3f}",
                        ha='center', va='bottom', fontsize=7, rotation=90,
                    )
        ax.set_xticks(x)
        ax.set_xticklabels(tasks, rotation=20, ha='right', fontsize=9)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.legend(title='Model', fontsize=8, title_fontsize=9,
                  loc='lower right' if higher_better else 'upper right',
                  framealpha=0.7)
        ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
        ax.yaxis.grid(True, linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)

    # ── Figure 1: Classification ───────────────────────────────────────────────
    if not classif_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Benchmark — Classification Tasks (val set, best horizon)",
                     fontsize=13, fontweight='bold', y=1.02)
        _bar_chart(axes[0], classif_df, 'val_auroc',
                   'AUROC by Task & Model', 'AUROC (higher is better)', higher_better=True)
        _bar_chart(axes[1], classif_df, 'val_auprc',
                   'AUPRC by Task & Model', 'AUPRC (higher is better)', higher_better=True)
        plt.tight_layout()
        out = os.path.join(WORK_DIR, "results", "classif_benchmark.png")
        plt.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved → {out}")
    else:
        print("No classification task results to plot.")

    # ── Figure 2: Regression ───────────────────────────────────────────────────
    if not regress_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Benchmark — Regression Tasks (val set, best horizon)",
                     fontsize=13, fontweight='bold', y=1.02)
        _bar_chart(axes[0], regress_df, 'val_rmse',
                   'RMSE by Task & Model', 'RMSE (lower is better)', higher_better=False)
        _bar_chart(axes[1], regress_df, 'val_mae',
                   'MAE by Task & Model',  'MAE (lower is better)',  higher_better=False)
        plt.tight_layout()
        out = os.path.join(WORK_DIR, "results", "regress_benchmark.png")
        plt.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved → {out}")
    else:
        print("No regression task results to plot.")